# Heavy Computation in a Web Worker

The JavaScript kernel runs inside a **Web Worker** — a background thread separate from the browser's main UI thread.

This means CPU-intensive code does not freeze the browser. While a heavy cell executes, you can still scroll the page, interact with other UI elements, and queue up the next cell.

This notebook runs four progressively heavier benchmarks, all off the main thread:

- **Sieve of Eratosthenes** — find all primes up to 200,000,000
- **Monte Carlo π estimation** — estimate π with 200 million random samples
- **Large array sort** — sort 20 million random floats
- **Matrix multiplication** — multiply two 1024×1024 matrices

> **Try it:** start running the cells and try scrolling this page — it stays fully responsive throughout.

## Setup

A shared `time()` helper measures each computation and collects results for a summary visualization at the end.

In [ ]:
// Shared results log — persists across all cells in this notebook
const results = [];

// Runs fn(), logs elapsed time, and stores the result
function time(label, fn) {
    const t0 = performance.now();
    const value = fn();
    const elapsed = +(performance.now() - t0).toFixed(1);
    results.push({ label, elapsed });
    console.log(`${label}: ${elapsed} ms`);
    return value;
}

## Sieve of Eratosthenes

A classic prime-finding algorithm: mark composite numbers by iterating over multiples of each prime found so far, then count what remains. Finding all primes up to 200 million requires marking hundreds of millions of entries in a typed array — a tight, cache-friendly inner loop with no pauses.

In [ ]:
function sieve(limit) {
    const composite = new Uint8Array(limit + 1); // composite[i] = 1 means i is not prime
    for (let i = 2; i * i <= limit; i++) {
        if (!composite[i]) {
            for (let j = i * i; j <= limit; j += i) {
                composite[j] = 1;
            }
        }
    }
    let count = 0;
    for (let i = 2; i <= limit; i++) {
        if (!composite[i]) count++;
    }
    return count;
}

const primeLimit = 200_000_000;
const primeCount = time(
    `Sieve of Eratosthenes (up to ${primeLimit.toLocaleString()})`,
    () => sieve(primeLimit)
);
console.log(`Found ${primeCount.toLocaleString()} primes`);
primeCount

## Monte Carlo π Estimation

Throw random darts at a unit square and count how many land inside the inscribed quarter-circle. The ratio converges to π/4. With 200 million samples the result is typically accurate to about 5 decimal places — and generating 400 million random numbers plus comparisons keeps the CPU busy for several seconds.

In [ ]:
function estimatePi(samples) {
    let inside = 0;
    for (let i = 0; i < samples; i++) {
        const x = Math.random();
        const y = Math.random();
        if (x * x + y * y <= 1) inside++;
    }
    return (4 * inside) / samples;
}

const piSamples = 200_000_000;
const piEstimate = time(
    `Monte Carlo π (${piSamples.toLocaleString()} samples)`,
    () => estimatePi(piSamples)
);
console.log(`π ≈ ${piEstimate.toFixed(8)}`);
console.log(`error: ${Math.abs(piEstimate - Math.PI).toFixed(8)}`);
piEstimate

## Sorting 20 Million Random Floats

Fill a `Float64Array` with random values, then sort it. JavaScript engines use TimSort (a hybrid merge/insertion sort), giving O(n log n) comparisons. For 20 million elements that is roughly 486 million comparison operations. `TypedArray.sort()` sorts numerically by default — no comparator needed.

In [ ]:
const sortSize = 20_000_000;

const sortedArr = time(
    `Sort ${sortSize.toLocaleString()} random floats`,
    () => {
        const arr = new Float64Array(sortSize);
        for (let i = 0; i < sortSize; i++) arr[i] = Math.random();
        arr.sort();
        return arr;
    }
);

console.log(`min: ${sortedArr[0].toFixed(8)}`);
console.log(`max: ${sortedArr[sortedArr.length - 1].toFixed(8)}`);
sortedArr.length

## Matrix Multiplication (1024×1024)

Multiplying two 1024×1024 matrices requires 1024³ ≈ 1.07 billion floating-point multiply-accumulate operations. The inner loop follows the i–k–j iteration order for better cache locality — reading `B` in sequential memory strides rather than the naive i–j–k column-jumping order.

In [ ]:
function matMul(a, b, n) {
    const c = new Float64Array(n * n);
    for (let i = 0; i < n; i++) {
        for (let k = 0; k < n; k++) {
            const aik = a[i * n + k];
            for (let j = 0; j < n; j++) {
                c[i * n + j] += aik * b[k * n + j];
            }
        }
    }
    return c;
}

const matSize = 1024;
const matA = Float64Array.from({ length: matSize * matSize }, Math.random);
const matB = Float64Array.from({ length: matSize * matSize }, Math.random);

const matC = time(
    `Matrix multiply ${matSize}×${matSize}`,
    () => matMul(matA, matB, matSize)
);

console.log(`C[0,0] = ${matC[0].toFixed(6)}`);
matC.length

## Results Summary

All four benchmarks ran entirely inside the Web Worker. The timings below are from your run — actual values vary by device and browser.

In [ ]:
const maxElapsed = Math.max(...results.map(r => r.elapsed));
const rows = results.map(({ label, elapsed }) => {
    const pct = (elapsed / maxElapsed * 90).toFixed(1);
    const bar = `<div style="background:linear-gradient(90deg,#4a90d9,#357abd);height:22px;width:${pct}%;border-radius:3px;min-width:2px"></div>`;
    return `<tr><td style="padding:7px 20px 7px 0;font-family:monospace;font-size:13px;white-space:nowrap;color:#333">${label}</td><td style="padding:7px 8px;min-width:260px;vertical-align:middle">${bar}</td><td style="padding:7px 0 7px 10px;font-family:monospace;font-size:13px;color:#555;white-space:nowrap">${elapsed} ms</td></tr>`;
}).join('');

`<div style="padding:20px 24px;background:#fafafa;border:1px solid #ddd;border-radius:8px;display:inline-block;font-family:sans-serif"><h3 style="margin:0 0 16px;font-size:15px;color:#222">Benchmark Results</h3><table style="border-collapse:collapse">${rows}</table></div>`